# Apply Building Mask

This notebook applies the building mask to the processed WorldView-3 multispectral mosaic.

If necessary, the building mask is automatically resampled to match the spatial resolution, extent, and coordinate reference system of the input raster. Pixels outside building footprints are assigned a NoData value while the original spectral values within buildings are preserved.

### Input

- Multispectral mosaic (`VNIR_SWIR_stack.tif`)
- Building mask (`Building_mask.tif`)

### Output

- Masked multispectral mosaic (`VNIR_SWIR_buildings.tif`)

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling


# Input and output files

data_dir = Path("../output")

mosaic_path = data_dir / "VNIR_SWIR_stack.tif"
mask_path = data_dir / "Building_mask.tif"

output_path = data_dir / "VNIR_SWIR_buildings.tif"


# Load input raster and building mask

with rasterio.open(mosaic_path) as src:
    image = src.read()
    profile = src.profile
    transform = src.transform
    crs = src.crs
    height = src.height
    width = src.width

    # Use the existing NoData value if available; otherwise define one.
    nodata_value = src.nodata if src.nodata is not None else -9999.0

with rasterio.open(mask_path) as src:
    mask = src.read(1)
    mask_transform = src.transform
    mask_crs = src.crs



# Resample the building mask to the mosaic grid

mask_resampled = np.zeros((height, width), dtype=np.uint8)

reproject(
    source=mask,
    destination=mask_resampled,
    src_transform=mask_transform,
    src_crs=mask_crs,
    dst_transform=transform,
    dst_crs=crs,
    resampling=Resampling.nearest,
)

# Apply building mask

# Preserve spectral values within buildings and assign NoData elsewhere.
masked_image = np.where(
    mask_resampled[np.newaxis, :, :] == 1,
    image,
    nodata_value,
).astype(np.float32)


profile.update(
    dtype="float32",
    count=image.shape[0],
    nodata=nodata_value,
    compress="lzw",
)


# Save masked mosaic

with rasterio.open(output_path, "w", **profile) as dst:
    dst.write(masked_image)

print(f"Masked mosaic saved as:\n{output_path.name}")